In [1]:
# read in ../results_highres_uk/{var}_mod[high/low]ice_highres_{scenario}_{member}_UK.nc
# get a list of site with lati and loni cover all UK sites
# for each variable and scenario,
# interately read in data of each member,
# for each site,
# get the value on each site[lati,loni]
# write in data with dims of [time,member] 
# finally, save data to here named {var}_{scenario}_UK_site{x}.res

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd

# 配置变量、场景、成员、站点列表
# vars = ["tas"]
scenarios = ["10000PGC"] #, "natural", "SSP126", "SSP245", "SSP370", "SSP585"] #,
vars = ["tas"]
scenarios = ["natural",  "SSP126", "SSP245", "SSP370", "SSP585","10000PGC"] #,
members = range(1, 91)  # 1~90
sites = pd.read_csv("UK_sites_index.res")  # 包含lat_index/lon_index/lat/lon等
output_dir = "./"
input_dir = "/user/work/bo20541/Downscaling_NWS/results_highres_uk"

# 如果var是pr，要读取PI的值，并且减去PI的值后再转换单位
# PI的值是/user/home/bo20541/Downscaling_NWS/training_data_highres/emul_in_Y_modhighice_pr.nc的第61个timestep
if vars[0] == "pr":
    pi_ds = xr.open_dataset("/user/home/bo20541/Downscaling_NWS/training_data_highres/emul_in_Y_modhighice_pr.nc")
    pi_value = pi_ds["var"][60,:,:].values  # 获取第61个timestep的值
    lat=pi_ds["lat"].values
    lon=pi_ds["lon"].values
    pi_ds.close()
    # print(f"PI value for pr: {pi_value}")
    print(f'shape of PI value: {pi_value.shape}, min: {pi_value.min()}, max: {pi_value.max()}')
if vars[0] == "tas":
    pi_ds = xr.open_dataset("/user/home/bo20541/Downscaling_NWS/training_data_highres/emul_in_Y_modhighice_tas.nc")
    pi_value = pi_ds["var"][60,:,:].values  # 获取第61个timestep的值
    lat=pi_ds["lat"].values
    lon=pi_ds["lon"].values
    pi_ds.close()
    # print(f"PI value for pr: {pi_value}")
    print(f'shape of PI value: {pi_value.shape}, min: {pi_value.min()}, max: {pi_value.max()}')

for var in vars:
    for scenario in scenarios:
        for site_idx, site_row in sites.iterrows():
            lati, loni = site_row['lat_index'], site_row['lon_index']
            lat_value,lon_value = site_row['lat'], site_row['lon']
            site_num = site_row['site_num']
            all_member_data = []
            for member in members:
                # ice high value
                fnamehigh = f"{var}_modhighice_highres_{scenario}_{member}_UK.nc"
                fpathhigh = os.path.join(input_dir, fnamehigh)
                dshigh = xr.open_dataset(fpathhigh)
                valhigh = dshigh["var"][:, int(lati), int(loni)]
                # ice low value
                fnameLow = f"{var}_modlowice_highres_{scenario}_{member}_UK.nc"
                fpathLow = os.path.join(input_dir, fnameLow)
                dsLow = xr.open_dataset(fpathLow)
                vallow = dsLow["var"][:, int(lati), int(loni)]
                # 读取对应 member 的 emul 输入文件中的 ice 列
                emul_fname = f"emul_inputs_{scenario}.{member}.updated.res"
                emul_fpath = os.path.join("/user/home/bo20541/Downscaling_NWS/SSP_forcings", emul_fname)
                emul_df = pd.read_csv(emul_fpath, sep=r"\s+", engine="python")
                ice_col = next((c for c in emul_df.columns if str(c).lower() == "ice"), None)
                ice = emul_df[ice_col].to_numpy()
                high_vals = valhigh.values
                low_vals = vallow.values

                if len(ice) != len(high_vals):
                    raise ValueError(
                        f"Length mismatch for member {member}: ice={len(ice)}, data={len(high_vals)}"
                    )

                # ice >= 0 用 lowice；否则(ice < 0)用 highice
                selected_vals = np.where(ice >= 0, low_vals, high_vals)
                all_member_data.append(selected_vals)
                dshigh.close()
                dsLow.close()
            # all_member_data: [member, time]
            all_member_data = np.stack(all_member_data)  # shape: [member, time]
            all_member_data = all_member_data.T  # shape: [time, member]
            if var == "tas":
                pi_site = pi_value[ilat, ilon]
                all_member_data = all_member_data - pi_site 
            if var == "pr":
                ilat = int(np.argmin(np.abs(lat - lat_value)))
                ilon = int(np.argmin(np.abs(lon - lon_value)))
                pi_site = pi_value[ilat, ilon]

                all_member_data = (all_member_data - pi_site) * 60.0 * 60.0 * 24.0 * 1000.0 * 30.0
            # 保存为 .txt 文件
            os.makedirs(output_dir+f"/site_{int(site_num)}", exist_ok=True)
            outname = f"{var}_{scenario}_UK_site{int(site_num)}.txt"
            np.savetxt(os.path.join(output_dir+f"/site_{int(site_num)}", outname), all_member_data)
            print(f"Saved {outname}, shape: {all_member_data.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '/user/home/bo20541/downscaling_NWS/training_data_highres/emul_in_Y_modhighice_pr.nc'